In [ ]:
import sys, os
import pickle
from pathlib import Path

def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    # search upward: handles CWD = notebooks/ or ascat_da/
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        # search downward one level: handles CWD = repo root
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')

_root = str(_find_lib_root())
if _root not in sys.path:
    sys.path.insert(0, _root)

_repo_root = Path(_root).parents[1]
_common_io = _repo_root / 'common' / 'python' / 'io'
if str(_common_io) not in sys.path:
    sys.path.insert(0, str(_common_io))
from read_GEOSldas import read_tilecoord, read_tilegrids

_scripts = Path(_root) / 'scripts'
if str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))
from check_raw_superobs_vs_ofa import read_ofa_species, collapse_ofa, summarize

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from datetime import datetime
from IPython.display import display

from lib.readers import read_bufr, read_h121, read_ofa
from lib.qc import QC_DEFAULT_BUFR, QC_DEFAULT_H121
from lib.superob import form_super_obs, match_tiles, ij_to_corners, EASE2, M36_CELL, X_ORIGIN, Y_ORIGIN


In [ ]:
# ── Configuration — edit here ─────────────────────────────────────────────────
DATES = [datetime(2020, 6, day) for day in range(1, 11)]
DATE  = datetime(2020, 6, 2)  # display day for single-day maps/distributions

# Bounding box (lat0, lon0, lat1, lon1)
DOMAIN = (35, -99, 38, -96)   # Oklahoma / S Kansas

# GEOSldas 3-hour analysis cycle labels scanned for spatial maps.
# 0=0000z, 1=0300z, ..., 7=2100z; obs are assigned to centered GEOS cycles.
PLOT_WINDOWS = range(8)
MIN_SPATIAL_MAP_OBS = 250      # plot raw-ob date/cycle only if one platform/product has at least this many obs
MAX_SPATIAL_MAP_FIGURES = None # set to an int while drafting if the notebook gets too long
MIN_REGIONAL_OFA_WINDOW_OBS = 50 # tile-count threshold for OFA zoom-window maps
MAX_REGIONAL_OFA_WINDOW_FIGURES = 30

# Data roots: local Discover sample, keeping the Discover Y????/M?? layout.
OBS_ROOT = '/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample'
BUFR_BASE = f'{OBS_ROOT}/legacy_bufr'
H121_BASE = f'{OBS_ROOT}/H121'
OFA_DIR   = ('/Users/amfox/Desktop/ASCAT_SSM_CDR/'
             'hsaf_cdr_test_DAv8_M36_202006_innov/output/SMAP_EASEv2_M36_GLOBAL/'
             'ana/ens_avg/Y2020/M06/')
TILE_BASE = f'{OBS_ROOT}/tilecoord/hsaf_cdr_test_DAv8_M36_202006_innov'
TILECOORD = f'{TILE_BASE}.ldas_tilecoord.bin'
TILEGRIDS = f'{TILE_BASE}.ldas_tilegrids.bin'

# Derived cache for raw reads + GEOS tile/cycle super-ob formation.
CACHE_VERSION = 'geos_cycle_v7_gridoriginfix_bsflag_noise'
CACHE_DIR = Path(_root) / '.cache' / 'legacy_vs_h121_obs'
GLOBAL_SUPEROB_VERSION = 'geos_cycle_global_v7_gridoriginfix_bsflag_noise'
GLOBAL_SUPEROB_CACHE_DIR = Path(_root) / '.cache' / 'global_superobs'
GLOBAL_OFA_TAG = f"20200601_20200610_{GLOBAL_SUPEROB_VERSION}"
OFA_CACHE_VERSION = 'v1'
OFA_CACHE_DIR = Path(_root) / '.cache' / 'ofa'
OFA_CACHE_TAG = f"20200601_20200610_{OFA_CACHE_VERSION}"
FORCE_REBUILD_CACHE = False


def month_dir(base, subdir, date):
    return f"{base}/{subdir}/Y{date:%Y}/M{date:%m}"

# QC — tweak individual keys to test sensitivity
QC_BUFR = {**QC_DEFAULT_BUFR}
QC_H121 = {**QC_DEFAULT_H121, 'sens_min': 1.0, 'subsfc_max': 5}

# Platforms
PLATFORMS = {
    'Metop-A': {'bufr_prefix': 'M02-ASCA-ASCSMO02-NA-5.0-', 'subdir': 'metop_a', 'color': '#1f77b4'},
    'Metop-B': {'bufr_prefix': 'M01-ASCA-ASCSMO02-NA-5.0-', 'subdir': 'metop_b', 'color': '#ff7f0e'},
    'Metop-C': {'bufr_prefix': 'M03-ASCA-ASCSMO02-NA-5.0-', 'subdir': 'metop_c', 'color': '#2ca02c'},
}

LAT0, LON0, LAT1, LON1 = DOMAIN
print(f"Domain: {LAT0}-{LAT1} deg N, {LON0}-{LON1} deg E")
print(f"Date range: {DATES[0].date()} through {DATES[-1].date()}   display day: {DATE.date()}")
print(f"Spatial-map GEOS cycle labels scanned: {list(PLOT_WINDOWS)}")
print(f"Spatial-map minimum obs threshold: {MIN_SPATIAL_MAP_OBS}")
print(f"Regional OFA window obs threshold: {MIN_REGIONAL_OFA_WINDOW_OBS}")
print(f"Obs root: {OBS_ROOT}")
print(f"Tilecoord: {TILECOORD}")
print(f"Regional cache dir: {CACHE_DIR}")
print(f"Global super-ob cache dir: {GLOBAL_SUPEROB_CACHE_DIR}")
print(f"OFA cache dir: {OFA_CACHE_DIR}")
print(f"H121 QC: {QC_H121}")


## 1. Start with GEOSldas ObsFcstAna

Before reproducing super-obs from raw files, first inspect what GEOSldas actually wrote to ObsFcstAna. The cache below is one row per analysis date/cycle/species/tile and includes OFA observation, forecast, analysis, innovation, increment, and count fields.


In [ ]:
# GEOS tile metadata from the experiment rc_out.
tile_coord = read_tilecoord(TILECOORD)
tile_grid_global, tile_grid_domain = read_tilegrids(TILEGRIDS)
print(f"Tilecoord: {tile_coord['N_tile']} tiles | grid={tile_grid_domain['gridtype'].strip()} {tile_grid_domain['N_lon']}x{tile_grid_domain['N_lat']}")

OFA_CACHE = OFA_CACHE_DIR / f'ofa_ascat_tile_cycle_{OFA_CACHE_TAG}.pkl'
OFA_SUMMARY = OFA_CACHE_DIR / f'ofa_ascat_tile_cycle_summary_{OFA_CACHE_TAG}.csv'
if not OFA_CACHE.exists():
    raise FileNotFoundError(
        'Missing OFA cache. Run: '
        'python projects/ascat_da/scripts/build_ofa_cache.py '
        '--start-date 2020-06-01 --end-date 2020-06-10 --version v1'
    )

ofa_cache = pd.read_pickle(OFA_CACHE)
ofa_cache_summary = pd.read_csv(OFA_SUMMARY) if OFA_SUMMARY.exists() else pd.DataFrame()
if len(ofa_cache_summary):
    display(ofa_cache_summary.round(3))
else:
    display(ofa_cache.groupby(['product', 'platform', 'species'])['n_ofa_obs'].sum().reset_index())

tile_meta = pd.DataFrame({
    'tilenum': np.asarray(tile_coord['tile_id'], dtype=np.int64),
    'tile_lat': np.asarray(tile_coord['com_lat'], dtype=float),
    'tile_lon': np.asarray(tile_coord['com_lon'], dtype=float),
    'tile_frac_cell': np.asarray(tile_coord['frac_cell'], dtype=float),
    'tile_area': np.asarray(tile_coord['area'], dtype=float),
    'tile_i': np.asarray(tile_coord['i_indg'], dtype=int),
    'tile_j': np.asarray(tile_coord['j_indg'], dtype=int),
})

ofa_tile_counts = (
    ofa_cache.groupby(['product', 'tilenum'], as_index=False)
    .agg(
        n_ofa_obs=('n_ofa_obs', 'sum'),
        mean_obs_pct=('obs_pct', 'mean'),
        mean_fcst_pct=('fcst_pct', 'mean'),
        mean_innov_pct=('innov_pct', 'mean'),
    )
    .merge(tile_meta, on='tilenum', how='left')
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6), subplot_kw={'projection': ccrs.Robinson()})
products = [('legacy', 'Legacy ASCAT OFA obs count'), ('h121', 'H121 ASCAT OFA obs count')]
vmax = max(1, ofa_tile_counts['n_ofa_obs'].max())
norm_count = mcolors.LogNorm(vmin=1, vmax=vmax)

for ax, (product, title) in zip(axes, products):
    p = ofa_tile_counts[ofa_tile_counts['product'] == product]
    ax.set_global()
    ax.set_facecolor('#f7f7f7')
    ax.gridlines(linewidth=0.25, color='#cccccc', alpha=0.7)
    sc = ax.scatter(
        p['tile_lon'], p['tile_lat'], c=p['n_ofa_obs'],
        s=2.2, cmap='viridis', norm=norm_count,
        transform=ccrs.PlateCarree(), linewidths=0, rasterized=True,
    )
    ax.set_title(f"{title}\n{DATES[0].date()} to {DATES[-1].date()} | {int(p['n_ofa_obs'].sum()):,} OFA obs")

cbar = fig.colorbar(sc, ax=axes, orientation='horizontal', fraction=0.05, pad=0.06)
cbar.set_label('OFA observations per GEOS tile over 10 days (log scale)')
fig.suptitle('GEOSldas ObsFcstAna ASCAT observation counts by tile', y=0.98)
plt.show()


## 2. OFA Legacy vs H121 comparisons

Compare what GEOSldas wrote to ObsFcstAna before looking at raw-ob reproduction. The first scatter uses 10-day OFA observation counts aggregated by tile. The second scatter uses matched OFA observation values by analysis date, cycle, platform, and tile.


In [ ]:
def _scatter_stats(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x = x[keep]
    y = y[keep]
    if len(x) == 0:
        return dict(n=0, bias=np.nan, rmse=np.nan, mae=np.nan, r=np.nan)
    diff = y - x
    return dict(
        n=len(x),
        bias=float(np.mean(diff)),
        rmse=float(np.sqrt(np.mean(diff ** 2))),
        mae=float(np.mean(np.abs(diff))),
        r=float(np.corrcoef(x, y)[0, 1]) if len(x) > 1 else np.nan,
    )


def _sample_for_plot(df, max_points=120000, random_state=42):
    if len(df) <= max_points:
        return df
    return df.sample(max_points, random_state=random_state)


# --- Count scatter: 10-day OFA observation counts per tile -------------------
count_panels = []
for plat in list(PLATFORMS) + ['All']:
    if plat == 'All':
        g = (
            ofa_cache.groupby(['product', 'tilenum'], as_index=False)
            .agg(n_ofa_obs=('n_ofa_obs', 'sum'))
        )
        keys = ['tilenum']
    else:
        g = (
            ofa_cache[ofa_cache['platform'] == plat]
            .groupby(['product', 'platform', 'tilenum'], as_index=False)
            .agg(n_ofa_obs=('n_ofa_obs', 'sum'))
        )
        keys = ['platform', 'tilenum']

    legacy_counts = g[g['product'] == 'legacy'][keys + ['n_ofa_obs']].rename(columns={'n_ofa_obs': 'legacy_count'})
    h121_counts = g[g['product'] == 'h121'][keys + ['n_ofa_obs']].rename(columns={'n_ofa_obs': 'h121_count'})
    pair = legacy_counts.merge(h121_counts, on=keys, how='outer').fillna(0)
    pair['panel'] = plat
    count_panels.append(pair)

ofa_count_pairs = pd.concat(count_panels, ignore_index=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharex=True, sharey=True)
panel_names = list(PLATFORMS) + ['All']
max_count = max(1, ofa_count_pairs[['legacy_count', 'h121_count']].to_numpy().max())
ref = np.array([0, max_count])

for ax, panel in zip(axes, panel_names):
    p = ofa_count_pairs[ofa_count_pairs['panel'] == panel]
    q = _sample_for_plot(p, max_points=100000)
    color = '0.25' if panel == 'All' else PLATFORMS[panel]['color']
    ax.scatter(q['legacy_count'], q['h121_count'], s=2, alpha=0.18, color=color, edgecolor='none')
    ax.plot(ref, ref, 'k--', lw=0.8)
    ax.set_xscale('symlog', linthresh=1)
    ax.set_yscale('symlog', linthresh=1)
    stats_count = _scatter_stats(p['legacy_count'], p['h121_count'])
    legacy_only = int(((p['legacy_count'] > 0) & (p['h121_count'] == 0)).sum())
    h121_only = int(((p['h121_count'] > 0) & (p['legacy_count'] == 0)).sum())
    txt = '\n'.join([
        f"tiles={len(p):,}",
        f"bias={stats_count['bias']:+.1f}",
        f"RMSE={stats_count['rmse']:.1f}",
        f"L-only={legacy_only:,}",
        f"H-only={h121_only:,}",
    ])
    ax.text(0.04, 0.96, txt, transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
    ax.set_title(panel)
    ax.set_xlabel('Legacy OFA obs count')
    ax.grid(True, linewidth=0.25, alpha=0.35)

axes[0].set_ylabel('H121 OFA obs count')
fig.suptitle(f'OFA observation-count comparison by tile | {DATES[0].date()} to {DATES[-1].date()}')
fig.tight_layout()
plt.show()


# --- Value scatter: matched OFA obs values by date/cycle/platform/tile --------
legacy_values = ofa_cache[ofa_cache['product'] == 'legacy'][
    ['date', 'cycle', 'platform', 'tilenum', 'obs_pct', 'fcst_pct', 'innov_pct', 'n_ofa_obs']
].rename(columns={
    'obs_pct': 'legacy_obs_pct',
    'fcst_pct': 'legacy_fcst_pct',
    'innov_pct': 'legacy_innov_pct',
    'n_ofa_obs': 'legacy_n_ofa_obs',
})
h121_values = ofa_cache[ofa_cache['product'] == 'h121'][
    ['date', 'cycle', 'platform', 'tilenum', 'obs_pct', 'fcst_pct', 'innov_pct', 'n_ofa_obs']
].rename(columns={
    'obs_pct': 'h121_obs_pct',
    'fcst_pct': 'h121_fcst_pct',
    'innov_pct': 'h121_innov_pct',
    'n_ofa_obs': 'h121_n_ofa_obs',
})

ofa_value_pairs = legacy_values.merge(
    h121_values,
    on=['date', 'cycle', 'platform', 'tilenum'],
    how='inner',
)
ofa_value_pairs['obs_diff_pct'] = ofa_value_pairs['h121_obs_pct'] - ofa_value_pairs['legacy_obs_pct']

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharex=True, sharey=True)
ref = np.array([0, 100])
for ax, panel in zip(axes, panel_names):
    p = ofa_value_pairs if panel == 'All' else ofa_value_pairs[ofa_value_pairs['platform'] == panel]
    q = _sample_for_plot(p, max_points=120000)
    color = '0.25' if panel == 'All' else PLATFORMS[panel]['color']
    ax.scatter(q['legacy_obs_pct'], q['h121_obs_pct'], s=2, alpha=0.15, color=color, edgecolor='none')
    ax.plot(ref, ref, 'k--', lw=0.8)
    stats_obs = _scatter_stats(p['legacy_obs_pct'], p['h121_obs_pct'])
    txt = '\n'.join([
        f"n={stats_obs['n']:,}",
        f"bias={stats_obs['bias']:+.2f}%",
        f"RMSE={stats_obs['rmse']:.2f}%",
        f"MAE={stats_obs['mae']:.2f}%",
        f"r={stats_obs['r']:.2f}",
    ])
    ax.text(0.04, 0.96, txt, transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
    ax.set_title(panel)
    ax.set_xlabel('Legacy OFA obs (% sat)')
    ax.set_aspect('equal')
    ax.grid(True, linewidth=0.25, alpha=0.35)

axes[0].set_ylabel('H121 OFA obs (% sat)')
for ax in axes:
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
fig.suptitle(f'OFA observation-value comparison on matched tile/cycles | {DATES[0].date()} to {DATES[-1].date()}')
fig.tight_layout()
plt.show()


# --- Probability distributions by sensor -------------------------------------
def _probability_hist(ax, df, value_col, bins, color, label):
    values = df[value_col].to_numpy(float)
    weights = df['n_ofa_obs'].to_numpy(float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if keep.sum() == 0:
        return
    weights = weights[keep] / weights[keep].sum()
    ax.hist(values[keep], bins=bins, weights=weights, histtype='step', lw=1.6,
            color=color, label=label)


def _weighted_summary(df, value_col):
    values = df[value_col].to_numpy(float)
    weights = df['n_ofa_obs'].to_numpy(float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if keep.sum() == 0:
        return np.nan, np.nan
    mean = np.average(values[keep], weights=weights[keep])
    mae = np.average(np.abs(values[keep]), weights=weights[keep])
    return float(mean), float(mae)


def _plot_ofa_probability_by_sensor(value_col, label, bins, xlim=None, summary='mean'):
    fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(15, 4.1), sharey=True)
    for ax, platform in zip(axes, PLATFORMS):
        for product, color, display_name in [
            ('legacy', '#4c78a8', 'Legacy'),
            ('h121', '#f58518', 'H121'),
        ]:
            p = ofa_cache[(ofa_cache['product'] == product) & (ofa_cache['platform'] == platform)]
            mean, mae = _weighted_summary(p, value_col)
            if summary == 'mae':
                legend_label = f'{display_name} | mean={mean:+.1f}, MAE={mae:.1f}'
            else:
                legend_label = f'{display_name} | mean={mean:.1f}'
            _probability_hist(ax, p, value_col, bins, color, legend_label)
        ax.set_title(platform)
        ax.set_xlabel(f'OFA {label} (% saturation)')
        if xlim is not None:
            ax.set_xlim(*xlim)
        ax.grid(True, linewidth=0.25, alpha=0.35)
        ax.legend(fontsize=8, frameon=False)
    axes[0].set_ylabel('Observation-weighted probability per bin')
    fig.suptitle(f'OFA {label} probability distributions by sensor | {DATES[0].date()} to {DATES[-1].date()}')
    fig.tight_layout()
    plt.show()


obs_bins = np.linspace(0, 100, 51)
innov_lim = np.nanpercentile(np.abs(ofa_cache['innov_pct']), 99.5)
innov_lim = min(80, max(20, float(innov_lim))) if np.isfinite(innov_lim) else 50
innov_bins = np.linspace(-innov_lim, innov_lim, 81)

_plot_ofa_probability_by_sensor('obs_pct', 'observation values', obs_bins, xlim=(0, 100))
_plot_ofa_probability_by_sensor('innov_pct', 'innovations', innov_bins, xlim=(-innov_lim, innov_lim), summary='mae')

# --- Maps: OFA fields by tile and sensor ------------------------------------
# These maps show whether the legacy/H121 scatter is spatially coherent.
def _weighted_mean(df, value_col, weight_col='tile_frac_cell'):
    vals = df[value_col].to_numpy(float)
    weights = df[weight_col].to_numpy(float) if weight_col in df else np.ones(len(df), float)
    keep = np.isfinite(vals) & np.isfinite(weights) & (weights > 0)
    if keep.sum() == 0:
        return np.nan
    return float(np.average(vals[keep], weights=weights[keep]))


def _plot_ofa_field_maps(value_col, label, agg='mean', cmap='viridis', value_range=None, diff_range=None):
    stat_col = f'{agg}_{value_col}'
    legacy_col = f'legacy_{stat_col}'
    h121_col = f'h121_{stat_col}'
    diff_col = f'h121_minus_legacy_{stat_col}'

    if agg == 'mean':
        agg_spec = {stat_col: (value_col, 'mean')}
        title_stat = 'mean'
    elif agg == 'mean_abs':
        source_col = f'abs_{value_col}'
        cache_for_agg = ofa_cache.assign(**{source_col: ofa_cache[value_col].abs()})
        agg_spec = {stat_col: (source_col, 'mean')}
        title_stat = 'mean absolute'
    elif agg == 'std':
        agg_spec = {stat_col: (value_col, 'std')}
        title_stat = 'std. dev.'
    else:
        raise ValueError(f'Unsupported aggregate: {agg}')
    agg_spec['n_ofa_obs'] = ('n_ofa_obs', 'sum')

    cache_for_agg = locals().get('cache_for_agg', ofa_cache)
    field = (
        cache_for_agg.groupby(['product', 'platform', 'tilenum'], as_index=False)
        .agg(**agg_spec)
        .merge(tile_meta, on='tilenum', how='left')
    )
    field = field[np.isfinite(field[stat_col])].copy()

    shared_cols = ['platform', 'tilenum', stat_col, 'n_ofa_obs', 'tile_lat', 'tile_lon', 'tile_frac_cell']
    legacy_field = field[field['product'] == 'legacy'][shared_cols].rename(
        columns={stat_col: legacy_col, 'n_ofa_obs': 'legacy_n_ofa_obs'}
    )
    h121_field = field[field['product'] == 'h121'][shared_cols].rename(
        columns={stat_col: h121_col, 'n_ofa_obs': 'h121_n_ofa_obs'}
    )
    pairs = legacy_field.merge(
        h121_field,
        on=['platform', 'tilenum', 'tile_lat', 'tile_lon', 'tile_frac_cell'],
        how='inner',
    )
    pairs[diff_col] = pairs[h121_col] - pairs[legacy_col]

    if value_range is None:
        vals = field[stat_col].to_numpy(float)
        lo, hi = np.nanpercentile(vals, [1, 99])
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo, hi = np.nanmin(vals), np.nanmax(vals)
        value_norm = mpl.colors.Normalize(vmin=float(lo), vmax=float(hi))
    else:
        value_norm = mpl.colors.Normalize(vmin=value_range[0], vmax=value_range[1])

    if diff_range is None:
        diff_lim = np.nanpercentile(np.abs(pairs[diff_col]), 98)
        diff_lim = max(1, float(diff_lim)) if np.isfinite(diff_lim) else 20
    else:
        diff_lim = float(diff_range)
    diff_norm = mpl.colors.TwoSlopeNorm(vmin=-diff_lim, vcenter=0, vmax=diff_lim)

    fig, axes = plt.subplots(
        len(PLATFORMS), 3, figsize=(16, 10.5),
        subplot_kw={'projection': ccrs.Robinson()},
        constrained_layout=True,
    )
    diff_cmap = 'RdBu_r'

    for row, platform in enumerate(PLATFORMS):
        legacy_p = field[(field['product'] == 'legacy') & (field['platform'] == platform)]
        h121_p = field[(field['product'] == 'h121') & (field['platform'] == platform)]
        diff_p = pairs[pairs['platform'] == platform]
        panels = [
            (f'Legacy {title_stat} {label}', legacy_p, stat_col, cmap, value_norm),
            (f'H121 {title_stat} {label}', h121_p, stat_col, cmap, value_norm),
            ('H121 - legacy', diff_p, diff_col, diff_cmap, diff_norm),
        ]
        for col, (title, data, plot_col, plot_cmap, norm) in enumerate(panels):
            ax = axes[row, col]
            ax.set_global()
            ax.set_facecolor('#f7f7f7')
            ax.gridlines(linewidth=0.25, color='0.72', alpha=0.35)
            ax.scatter(
                data['tile_lon'], data['tile_lat'], c=data[plot_col],
                s=1.6, cmap=plot_cmap, norm=norm,
                transform=ccrs.PlateCarree(), linewidths=0, rasterized=True,
            )
            wmean = _weighted_mean(data, plot_col)
            if row == 0:
                ax.set_title(title)
            ax.text(0.02, 0.04, f'land-wtd mean={wmean:+.2f}', transform=ax.transAxes,
                    ha='left', va='bottom', fontsize=8,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, pad=2))
            if col == 0:
                ax.text(-0.03, 0.5, platform, transform=ax.transAxes, rotation=90,
                        va='center', ha='right', fontsize=11, fontweight='bold')

    cbar_value = fig.colorbar(
        mpl.cm.ScalarMappable(norm=value_norm, cmap=cmap),
        ax=axes[:, :2], orientation='horizontal', fraction=0.035, pad=0.035,
    )
    cbar_value.set_label(f'OFA {title_stat} {label} (% saturation)')
    cbar_diff = fig.colorbar(
        mpl.cm.ScalarMappable(norm=diff_norm, cmap=diff_cmap),
        ax=axes[:, 2], orientation='horizontal', fraction=0.035, pad=0.035,
    )
    cbar_diff.set_label(f'H121 - legacy OFA {title_stat} {label} (% saturation)')
    fig.suptitle(f'OFA {title_stat} {label} by GEOS tile | {DATES[0].date()} to {DATES[-1].date()}', y=1.02)
    plt.show()
    return field, pairs


ofa_mean_obs, ofa_mean_obs_pairs = _plot_ofa_field_maps(
    'obs_pct', 'observation values', agg='mean', value_range=(0, 100), diff_range=None,
)
ofa_mean_fcst, ofa_mean_fcst_pairs = _plot_ofa_field_maps(
    'fcst_pct', 'forecast values', agg='mean', value_range=(0, 100), diff_range=None,
)
ofa_mean_innov, ofa_mean_innov_pairs = _plot_ofa_field_maps(
    'innov_pct', 'innovations', agg='mean', cmap='RdBu_r', value_range=None, diff_range=None,
)
ofa_mean_abs_innov, ofa_mean_abs_innov_pairs = _plot_ofa_field_maps(
    'innov_pct', 'absolute innovations', agg='mean_abs', cmap='magma', value_range=(0, 40), diff_range=None,
)
ofa_std_innov, ofa_std_innov_pairs = _plot_ofa_field_maps(
    'innov_pct', 'innovation standard deviation', agg='std', cmap='magma', value_range=(0, 35), diff_range=None,
)


## 3. Regional OFA diagnostics

Repeat the ObsFcstAna-only checks over the configured zoom domain before moving back to raw-ob/super-ob reproduction. The window maps use OFA tile values only, so each selected platform/cycle is shown as a two-panel legacy/H121 tile comparison.


In [ ]:
# Regional subset of the OFA tile-cycle cache.
region_tiles = tile_meta[
    tile_meta['tile_lat'].between(LAT0, LAT1) & tile_meta['tile_lon'].between(LON0, LON1)
].copy()
region_tile_cols = ['tilenum', 'tile_lat', 'tile_lon', 'tile_frac_cell', 'tile_i', 'tile_j']
ofa_region = (
    ofa_cache[ofa_cache['tilenum'].isin(region_tiles['tilenum'])]
    .drop(columns=[c for c in region_tile_cols[1:] if c in ofa_cache.columns], errors='ignore')
    .merge(tile_meta[region_tile_cols], on='tilenum', how='left')
)
print(f'Regional OFA rows: {len(ofa_region):,} | tiles with OFA: {ofa_region.tilenum.nunique():,} | domain={LAT0}-{LAT1}N, {LON0}-{LON1}E')


def _setup_region_ax(ax, title=None):
    pad = 0.35
    ax.set_extent([LON0 - pad, LON1 + pad, LAT0 - pad, LAT1 + pad], crs=ccrs.PlateCarree())
    ax.set_facecolor('#f7f7f7')
    ax.gridlines(draw_labels=False, linewidth=0.25, color='0.70', alpha=0.45)
    if title:
        ax.set_title(title)


def _draw_tile_values(ax, data, value_col, cmap, norm, *, edgecolor='#555', linewidth=0.15):
    data = data[np.isfinite(data[value_col])].copy()
    if len(data) == 0:
        return None
    patches = []
    values = []
    for _, r in data.iterrows():
        clons, clats = ij_to_corners(int(r['tile_i']), int(r['tile_j']))
        patches.append(mpatches.Polygon(np.column_stack([clons, clats]), closed=True))
        values.append(float(r[value_col]))
    coll = PatchCollection(
        patches, cmap=cmap, norm=norm, edgecolor=edgecolor, linewidth=linewidth,
        transform=ccrs.PlateCarree(), zorder=4, rasterized=True,
    )
    coll.set_array(np.asarray(values, dtype=float))
    ax.add_collection(coll)
    return coll


# --- Regional OFA count maps -------------------------------------------------
region_counts = (
    ofa_region.groupby(['product', 'tilenum'], as_index=False)
    .agg(n_ofa_obs=('n_ofa_obs', 'sum'))
    .merge(tile_meta[region_tile_cols], on='tilenum', how='left')
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), subplot_kw={'projection': ccrs.PlateCarree()})
count_norm = mpl.colors.LogNorm(vmin=1, vmax=max(1, region_counts['n_ofa_obs'].max()))
for ax, product, title in zip(axes, ['legacy', 'h121'], ['Legacy', 'H121']):
    p = region_counts[region_counts['product'] == product]
    _setup_region_ax(ax, f'{title} OFA obs count')
    _draw_tile_values(ax, p, 'n_ofa_obs', plt.cm.viridis, count_norm)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=count_norm, cmap='viridis'), ax=axes, orientation='horizontal', fraction=0.055, pad=0.08)
cbar.set_label('OFA observations per tile over 10 days (log scale)')
fig.suptitle(f'Regional OFA observation counts | {DATES[0].date()} to {DATES[-1].date()}')
plt.show()


# --- Regional scatter plots --------------------------------------------------
def _regional_count_pairs(data):
    panels = []
    for plat in list(PLATFORMS) + ['All']:
        if plat == 'All':
            g = data.groupby(['product', 'tilenum'], as_index=False).agg(n_ofa_obs=('n_ofa_obs', 'sum'))
            keys = ['tilenum']
        else:
            g = data[data['platform'] == plat].groupby(['product', 'platform', 'tilenum'], as_index=False).agg(n_ofa_obs=('n_ofa_obs', 'sum'))
            keys = ['platform', 'tilenum']
        l = g[g['product'] == 'legacy'][keys + ['n_ofa_obs']].rename(columns={'n_ofa_obs': 'legacy_count'})
        h = g[g['product'] == 'h121'][keys + ['n_ofa_obs']].rename(columns={'n_ofa_obs': 'h121_count'})
        pair = l.merge(h, on=keys, how='outer').fillna(0)
        pair['panel'] = plat
        panels.append(pair)
    return pd.concat(panels, ignore_index=True)

region_count_pairs = _regional_count_pairs(ofa_region)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.0), sharex=True, sharey=True)
max_count = max(1, region_count_pairs[['legacy_count', 'h121_count']].to_numpy().max())
ref = np.array([0, max_count])
for ax, panel in zip(axes, panel_names):
    p = region_count_pairs[region_count_pairs['panel'] == panel]
    color = '0.25' if panel == 'All' else PLATFORMS[panel]['color']
    ax.scatter(p['legacy_count'], p['h121_count'], s=12, alpha=0.45, color=color, edgecolor='none')
    ax.plot(ref, ref, 'k--', lw=0.8)
    ax.set_xscale('symlog', linthresh=1)
    ax.set_yscale('symlog', linthresh=1)
    st = _scatter_stats(p['legacy_count'], p['h121_count'])
    ax.text(0.04, 0.96, f"tiles={len(p):,}\nbias={st['bias']:+.1f}\nRMSE={st['rmse']:.1f}", transform=ax.transAxes,
            va='top', fontsize=8, bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
    ax.set_title(panel)
    ax.set_xlabel('Legacy count')
    ax.grid(True, linewidth=0.25, alpha=0.35)
axes[0].set_ylabel('H121 count')
fig.suptitle('Regional OFA observation-count comparison')
fig.tight_layout()
plt.show()

region_value_pairs = (
    ofa_region[ofa_region['product'] == 'legacy'][['date', 'cycle', 'platform', 'tilenum', 'obs_pct', 'innov_pct', 'n_ofa_obs']]
    .rename(columns={'obs_pct': 'legacy_obs_pct', 'innov_pct': 'legacy_innov_pct', 'n_ofa_obs': 'legacy_n_ofa_obs'})
    .merge(
        ofa_region[ofa_region['product'] == 'h121'][['date', 'cycle', 'platform', 'tilenum', 'obs_pct', 'innov_pct', 'n_ofa_obs']]
        .rename(columns={'obs_pct': 'h121_obs_pct', 'innov_pct': 'h121_innov_pct', 'n_ofa_obs': 'h121_n_ofa_obs'}),
        on=['date', 'cycle', 'platform', 'tilenum'], how='inner'
    )
)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.0), sharex=True, sharey=True)
ref = np.array([0, 100])
for ax, panel in zip(axes, panel_names):
    p = region_value_pairs if panel == 'All' else region_value_pairs[region_value_pairs['platform'] == panel]
    color = '0.25' if panel == 'All' else PLATFORMS[panel]['color']
    ax.scatter(p['legacy_obs_pct'], p['h121_obs_pct'], s=9, alpha=0.35, color=color, edgecolor='none')
    ax.plot(ref, ref, 'k--', lw=0.8)
    st = _scatter_stats(p['legacy_obs_pct'], p['h121_obs_pct'])
    ax.text(0.04, 0.96, f"n={st['n']:,}\nbias={st['bias']:+.2f}\nRMSE={st['rmse']:.2f}", transform=ax.transAxes,
            va='top', fontsize=8, bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
    ax.set_title(panel)
    ax.set_xlabel('Legacy obs (% sat)')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_aspect('equal')
    ax.grid(True, linewidth=0.25, alpha=0.35)
axes[0].set_ylabel('H121 obs (% sat)')
fig.suptitle('Regional OFA observation-value comparison on matched tile/cycles')
fig.tight_layout()
plt.show()


# --- Regional probability plots ---------------------------------------------
def _plot_regional_probability_by_sensor(data, value_col, label, bins, xlim=None, summary='mean'):
    fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(15, 4.0), sharey=True)
    for ax, platform in zip(axes, PLATFORMS):
        for product, color, display_name in [('legacy', '#4c78a8', 'Legacy'), ('h121', '#f58518', 'H121')]:
            p = data[(data['product'] == product) & (data['platform'] == platform)]
            mean, mae = _weighted_summary(p, value_col)
            label_text = f'{display_name} | mean={mean:+.1f}, MAE={mae:.1f}' if summary == 'mae' else f'{display_name} | mean={mean:.1f}'
            _probability_hist(ax, p, value_col, bins, color, label_text)
        ax.set_title(platform)
        ax.set_xlabel(f'OFA {label} (% saturation)')
        if xlim is not None:
            ax.set_xlim(*xlim)
        ax.grid(True, linewidth=0.25, alpha=0.35)
        ax.legend(fontsize=8, frameon=False)
    axes[0].set_ylabel('Observation-weighted probability per bin')
    fig.suptitle(f'Regional OFA {label} probability distributions by sensor')
    fig.tight_layout()
    plt.show()

_plot_regional_probability_by_sensor(ofa_region, 'obs_pct', 'observation values', obs_bins, xlim=(0, 100))
_plot_regional_probability_by_sensor(ofa_region, 'innov_pct', 'innovations', innov_bins, xlim=(-innov_lim, innov_lim), summary='mae')


# --- Regional mean-field maps ------------------------------------------------
def _plot_regional_ofa_field_maps(data, value_col, label, agg='mean', cmap='viridis', value_range=None, diff_range=None):
    stat_col = f'{agg}_{value_col}'
    legacy_col = f'legacy_{stat_col}'
    h121_col = f'h121_{stat_col}'
    diff_col = f'h121_minus_legacy_{stat_col}'
    cache_for_agg = data
    if agg == 'mean':
        agg_spec = {stat_col: (value_col, 'mean')}
        title_stat = 'mean'
    elif agg == 'mean_abs':
        source_col = f'abs_{value_col}'
        cache_for_agg = data.assign(**{source_col: data[value_col].abs()})
        agg_spec = {stat_col: (source_col, 'mean')}
        title_stat = 'mean absolute'
    elif agg == 'std':
        agg_spec = {stat_col: (value_col, 'std')}
        title_stat = 'std. dev.'
    else:
        raise ValueError(agg)
    agg_spec['n_ofa_obs'] = ('n_ofa_obs', 'sum')
    field = cache_for_agg.groupby(['product', 'platform', 'tilenum'], as_index=False).agg(**agg_spec)
    field = field.merge(tile_meta[region_tile_cols], on='tilenum', how='left')
    field = field[np.isfinite(field[stat_col])].copy()
    shared = ['platform', 'tilenum', stat_col, 'n_ofa_obs', 'tile_lat', 'tile_lon', 'tile_frac_cell', 'tile_i', 'tile_j']
    l = field[field['product'] == 'legacy'][shared].rename(columns={stat_col: legacy_col, 'n_ofa_obs': 'legacy_n_ofa_obs'})
    h = field[field['product'] == 'h121'][shared].rename(columns={stat_col: h121_col, 'n_ofa_obs': 'h121_n_ofa_obs'})
    pairs = l.merge(h, on=['platform', 'tilenum', 'tile_lat', 'tile_lon', 'tile_frac_cell', 'tile_i', 'tile_j'], how='inner')
    pairs[diff_col] = pairs[h121_col] - pairs[legacy_col]
    if value_range is None:
        lo, hi = np.nanpercentile(field[stat_col], [1, 99])
        value_norm = mpl.colors.Normalize(vmin=float(lo), vmax=float(hi))
    else:
        value_norm = mpl.colors.Normalize(vmin=value_range[0], vmax=value_range[1])
    if diff_range is None:
        diff_lim = np.nanpercentile(np.abs(pairs[diff_col]), 98)
        diff_lim = max(1, float(diff_lim)) if np.isfinite(diff_lim) else 20
    else:
        diff_lim = float(diff_range)
    diff_norm = mpl.colors.TwoSlopeNorm(vmin=-diff_lim, vcenter=0, vmax=diff_lim)

    fig, axes = plt.subplots(len(PLATFORMS), 3, figsize=(13, 10), subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
    for row, platform in enumerate(PLATFORMS):
        legacy_p = field[(field['product'] == 'legacy') & (field['platform'] == platform)]
        h121_p = field[(field['product'] == 'h121') & (field['platform'] == platform)]
        diff_p = pairs[pairs['platform'] == platform]
        panels = [
            (f'Legacy {title_stat} {label}', legacy_p, stat_col, cmap, value_norm),
            (f'H121 {title_stat} {label}', h121_p, stat_col, cmap, value_norm),
            ('H121 - legacy', diff_p, diff_col, 'RdBu_r', diff_norm),
        ]
        for col, (title, d, plot_col, plot_cmap, norm) in enumerate(panels):
            ax = axes[row, col]
            _setup_region_ax(ax, title if row == 0 else None)
            _draw_tile_values(ax, d, plot_col, plt.get_cmap(plot_cmap), norm)
            ax.text(0.02, 0.04, f'land-wtd mean={_weighted_mean(d, plot_col):+.2f}', transform=ax.transAxes,
                    ha='left', va='bottom', fontsize=8, bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, pad=2))
            if col == 0:
                ax.text(-0.06, 0.5, platform, transform=ax.transAxes, rotation=90,
                        va='center', ha='right', fontsize=11, fontweight='bold')
    fig.colorbar(mpl.cm.ScalarMappable(norm=value_norm, cmap=cmap), ax=axes[:, :2], orientation='horizontal', fraction=0.04, pad=0.05).set_label(f'OFA {title_stat} {label} (% saturation)')
    fig.colorbar(mpl.cm.ScalarMappable(norm=diff_norm, cmap='RdBu_r'), ax=axes[:, 2], orientation='horizontal', fraction=0.04, pad=0.05).set_label(f'H121 - legacy OFA {title_stat} {label} (% saturation)')
    fig.suptitle(f'Regional OFA {title_stat} {label} by GEOS tile')
    plt.show()
    return field, pairs

region_mean_obs, region_mean_obs_pairs = _plot_regional_ofa_field_maps(ofa_region, 'obs_pct', 'observation values', agg='mean', value_range=(0, 100))
region_mean_fcst, region_mean_fcst_pairs = _plot_regional_ofa_field_maps(ofa_region, 'fcst_pct', 'forecast values', agg='mean', value_range=(0, 100))
region_mean_innov, region_mean_innov_pairs = _plot_regional_ofa_field_maps(ofa_region, 'innov_pct', 'innovations', agg='mean', cmap='RdBu_r')
region_mean_abs_innov, region_mean_abs_innov_pairs = _plot_regional_ofa_field_maps(ofa_region, 'innov_pct', 'absolute innovations', agg='mean_abs', cmap='magma', value_range=(0, 40))
region_std_innov, region_std_innov_pairs = _plot_regional_ofa_field_maps(ofa_region, 'innov_pct', 'innovation standard deviation', agg='std', cmap='magma', value_range=(0, 35))


# --- Regional date/cycle window maps -----------------------------------------
# One figure per useful GEOS analysis cycle: rows are sensors, columns are products.
window_counts = (
    ofa_region.groupby(['date', 'cycle', 'product'], as_index=False)
    .agg(n_ofa_obs=('n_ofa_obs', 'sum'))
    .pivot_table(index=['date', 'cycle'], columns='product', values='n_ofa_obs', fill_value=0)
    .reset_index()
)
for product in ['legacy', 'h121']:
    if product not in window_counts:
        window_counts[product] = 0
window_counts['max_product_obs'] = window_counts[['legacy', 'h121']].max(axis=1)
region_windows = window_counts[window_counts['max_product_obs'] >= MIN_REGIONAL_OFA_WINDOW_OBS].sort_values(['date', 'cycle'])
if MAX_REGIONAL_OFA_WINDOW_FIGURES is not None:
    region_windows = region_windows.head(MAX_REGIONAL_OFA_WINDOW_FIGURES)
print(f'Regional OFA window maps: plotting {len(region_windows)} date/cycle windows with >= {MIN_REGIONAL_OFA_WINDOW_OBS} obs in at least one product.')


def _add_corner_latlon_labels(ax):
    ax.set_xticks([LON0, LON1], crs=ccrs.PlateCarree())
    ax.set_yticks([LAT0, LAT1], crs=ccrs.PlateCarree())
    ax.set_xticklabels([f'{abs(LON0):.0f}W', f'{abs(LON1):.0f}W'])
    ax.set_yticklabels([f'{LAT0:.0f}N', f'{LAT1:.0f}N'])
    ax.tick_params(labelsize=8, length=3)


for _, win in region_windows.iterrows():
    date = pd.Timestamp(win['date']).date()
    cycle = int(win['cycle'])
    subset = ofa_region[(pd.to_datetime(ofa_region['date']).dt.date == date) & (ofa_region['cycle'] == cycle)]

    fig, axes = plt.subplots(
        len(PLATFORMS), 2, figsize=(10.5, 10.0),
        subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True,
    )
    axes = np.atleast_2d(axes)
    for row, platform in enumerate(PLATFORMS):
        for col, (product, product_title) in enumerate([('legacy', 'Legacy OFA obs'), ('h121', 'H121 OFA obs')]):
            ax = axes[row, col]
            p = subset[(subset['platform'] == platform) & (subset['product'] == product)]
            n_obs = int(p.n_ofa_obs.sum()) if len(p) else 0
            title = product_title if row == 0 else None
            _setup_region_ax(ax, title)
            _draw_tile_values(ax, p, 'obs_pct', plt.cm.viridis, mpl.colors.Normalize(0, 100))
            ax.text(0.02, 0.96, f'n={n_obs}', transform=ax.transAxes,
                    ha='left', va='top', fontsize=8,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
            if col == 0:
                ax.text(-0.08, 0.5, platform, transform=ax.transAxes, rotation=90,
                        va='center', ha='right', fontsize=10, fontweight='bold')
    _add_corner_latlon_labels(axes[-1, 0])
    fig.colorbar(
        mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 100), cmap='viridis'),
        ax=axes, orientation='horizontal', fraction=0.035, pad=0.04,
    ).set_label('OFA obs (% saturation)')
    fig.suptitle(f'{date} cycle {cycle:02d} | regional OFA tile obs')
    plt.show()
    plt.close(fig)


## 4. Load/cache raw obs and GEOS tile-cycle super-obs

After looking at OFA, read the raw Legacy BUFR and H121 sample, assign observations to the experiment GEOS `tilecoord`, and form super-obs by `tilenum + cycle`. The derived raw/super-ob payloads are cached locally so rerunning the notebook does not repeatedly parse BUFR and H121 files.


In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def domain_tag(domain):
    return '_'.join(str(x).replace('-', 'm').replace('.', 'p') for x in domain)


def cache_file(date):
    return CACHE_DIR / f"superobs_{date:%Y%m%d}_{CACHE_VERSION}_{domain_tag(DOMAIN)}.pkl"


def load_day(date, force=False):
    fcache = cache_file(date)
    if fcache.exists() and not force:
        with fcache.open('rb') as f:
            payload = pickle.load(f)
        print(f"{date:%Y-%m-%d}: loaded cache")
        return payload

    raw_day = {}
    so_day = {}
    count_rows = []
    for plat, cfg in PLATFORMS.items():
        bdir = month_dir(BUFR_BASE, cfg['subdir'], date)
        hdir = month_dir(H121_BASE, cfg['subdir'], date)
        b = read_bufr(bdir, date, cfg['bufr_prefix'], domain=DOMAIN, qc=QC_BUFR)
        h = read_h121(hdir, date, domain=DOMAIN, qc=QC_H121)
        raw_day[plat] = {'bufr': b, 'h121': h}
        so_day[plat] = {
            'bufr_pool': form_super_obs(b['lat'], b['lon'], b['ssm'], tile_coord=tile_coord, tile_grid=tile_grid_domain),
            'h121_pool': form_super_obs(h['lat'], h['lon'], h['ssm'], tile_coord=tile_coord, tile_grid=tile_grid_domain),
            'bufr_win':  form_super_obs(b['lat'], b['lon'], b['ssm'], window=b['window'], cycle=b['cycle'], tile_coord=tile_coord, tile_grid=tile_grid_domain),
            'h121_win':  form_super_obs(h['lat'], h['lon'], h['ssm'], window=h['window'], cycle=h['cycle'], tile_coord=tile_coord, tile_grid=tile_grid_domain),
        }
        count_rows.append({
            'date': date.date(),
            'platform': plat,
            'legacy_obs': len(b['ssm']),
            'h121_obs': len(h['ssm']),
            'legacy_tiles': len(so_day[plat]['bufr_pool']['ssm']),
            'h121_tiles': len(so_day[plat]['h121_pool']['ssm']),
            'legacy_tile_cycles': len(so_day[plat]['bufr_win']['ssm']),
            'h121_tile_cycles': len(so_day[plat]['h121_win']['ssm']),
        })

    payload = {'date': date, 'raw': raw_day, 'so': so_day, 'counts': pd.DataFrame(count_rows)}
    with fcache.open('wb') as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"{date:%Y-%m-%d}: read obs and wrote cache")
    return payload


daily = {date: load_day(date, force=FORCE_REBUILD_CACHE) for date in DATES}

# Single display day used by later one-day maps/distributions.
raw = daily[DATE]['raw']
so = daily[DATE]['so']

counts = pd.concat([payload['counts'] for payload in daily.values()], ignore_index=True)
display(counts.groupby('platform')[['legacy_obs', 'h121_obs', 'legacy_tile_cycles', 'h121_tile_cycles']].sum())


## 5. Global validation of raw super-obs against GEOSldas ObsFcstAna

Before comparing Legacy BUFR and H121 as products, verify that the raw-ob processing is consistent with the GEOSldas observation stream in global tile/cycle space. This section uses the derived global super-ob cache and the global OFA validation outputs from `check_global_superobs_vs_ofa.py`; it does not reread raw BUFR/H121 files. Raw-only tile/cycles are expected because GEOSldas applies additional model-based screening.


In [ ]:
OFA_SPECIES = {
    'legacy': {'Metop-A': 9,  'Metop-B': 10, 'Metop-C': 11},
    'h121':   {'Metop-A': 14, 'Metop-B': 15, 'Metop-C': 16},
}

GLOBAL_OFA_SUMMARY = GLOBAL_SUPEROB_CACHE_DIR / f'global_superobs_vs_ofa_summary_{GLOBAL_OFA_TAG}.csv'
GLOBAL_OFA_PAIRS = GLOBAL_SUPEROB_CACHE_DIR / f'global_superobs_vs_ofa_pairs_{GLOBAL_OFA_TAG}.csv'

if not GLOBAL_OFA_SUMMARY.exists() or not GLOBAL_OFA_PAIRS.exists():
    raise FileNotFoundError(
        'Missing global OFA validation outputs. Run: '
        'python projects/ascat_da/scripts/check_global_superobs_vs_ofa.py '
        '--start-date 2020-06-01 --end-date 2020-06-10'
    )

ofa_validation = pd.read_csv(GLOBAL_OFA_SUMMARY)
num_cols = ['matched_frac_ofa', 'bias_pct', 'rmse_pct', 'mae_pct', 'median_abs_pct', 'max_abs_pct']
ofa_validation[num_cols] = ofa_validation[num_cols].round(3)
display(ofa_validation)

# The matched-pairs CSV is large, so sample for plotting while keeping exact
# statistics from the summary table above.
PAIR_COLUMNS = [
    'date', 'product', 'platform', 'species', 'tilenum', 'cycle', 'window',
    'ssm_pct', 'ofa_obs_pct', 'diff_pct', 'n_obs', 'ofa_count', 'ssm_std_pct',
]
PLOT_SAMPLE_PER_PANEL = 60000

sample_parts = []
for chunk in pd.read_csv(GLOBAL_OFA_PAIRS, usecols=PAIR_COLUMNS, chunksize=250000):
    for product in ['legacy', 'h121']:
        for plat in PLATFORMS:
            part = chunk[(chunk['product'] == product) & (chunk['platform'] == plat)]
            if len(part):
                sample_parts.append(part.sample(
                    min(len(part), max(1000, PLOT_SAMPLE_PER_PANEL // 6)),
                    random_state=42,
                ))

ofa_pairs_sample = pd.concat(sample_parts, ignore_index=True)
# Resample to a stable cap per panel after chunk-wise sampling.
ofa_pairs_plot = []
for (product, plat), group in ofa_pairs_sample.groupby(['product', 'platform']):
    ofa_pairs_plot.append(group.sample(min(len(group), PLOT_SAMPLE_PER_PANEL), random_state=42))
ofa_pairs_plot = pd.concat(ofa_pairs_plot, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.4), sharex=True, sharey=True)
ref = np.array([0, 100])
for row, product in enumerate(['legacy', 'h121']):
    for col, (plat, cfg) in enumerate(PLATFORMS.items()):
        ax = axes[row, col]
        p = ofa_pairs_plot[(ofa_pairs_plot['product'] == product) & (ofa_pairs_plot['platform'] == plat)]
        s = ofa_validation[(ofa_validation['product'] == product) & (ofa_validation['platform'] == plat)].iloc[0]
        ax.plot(ref, ref, 'k--', lw=0.8, zorder=1)
        if len(p):
            ax.scatter(p['ssm_pct'], p['ofa_obs_pct'], s=2, alpha=0.18,
                       color=cfg['color'], edgecolor='none', zorder=2)
        txt = '\n'.join([
            f"matched={int(s['matched']):,}",
            f"OFA frac={s['matched_frac_ofa']:.3f}",
            f"bias={s['bias_pct']:+.2f}%",
            f"RMSE={s['rmse_pct']:.2f}%",
            f"med abs={s['median_abs_pct']:.2f}%",
        ])
        ax.text(0.04, 0.96, txt, transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))
        ax.set_title(f"{product} {plat}\nOFA species {OFA_SPECIES[product][plat]}", fontsize=10)
        ax.set_aspect('equal')
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        if row == 1:
            ax.set_xlabel('Cached raw super-ob (% sat)')
        if col == 0:
            ax.set_ylabel('ObsFcstAna obs (% sat)')

fig.suptitle(f'Global raw super-obs vs GEOSldas ObsFcstAna | {DATES[0].date()} to {DATES[-1].date()} | sampled points, exact stats')
fig.tight_layout()
plt.show()


## 6. SSM distributions


In [ ]:
fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(14, 4), sharey=True)
bins = np.linspace(0, 100, 41)

for ax, (plat, d) in zip(axes, raw.items()):
    b, h = d['bufr'], d['h121']
    ax.hist(b['ssm'], bins=bins, alpha=0.7, label=f'Legacy (n={len(b["ssm"])})', color='steelblue')
    ax.hist(h['ssm'], bins=bins, alpha=0.5, label=f'H121 (n={len(h["ssm"])})', color='#e07b00')
    ax.set_title(plat)
    ax.set_xlabel('SSM (% saturation)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Count')
fig.suptitle(f'Raw obs SSM distributions  |  {DATE.date()}  |  {LAT0}–{LAT1}°N, {LON0}–{LON1}°E')
fig.tight_layout()
plt.show()


## 7. Spatial maps

Scan all 10 days and make zoom-in maps only for date/cycle windows with enough observations in the domain. A window is plotted when at least one platform/product has `MIN_SPATIAL_MAP_OBS` raw observations in the zoom box.


In [ ]:
def draw_m36_grid(ax):
    """Draw M36 EASE cell boundaries within the domain."""
    margin = 0.3
    xs = [EASE2(LON0 - margin, LAT0 - margin)[0], EASE2(LON1 + margin, LAT1 + margin)[0]]
    ys = [EASE2(LON0 - margin, LAT0 - margin)[1], EASE2(LON1 + margin, LAT1 + margin)[1]]
    x_lo, x_hi = min(xs) - M36_CELL, max(xs) + M36_CELL
    y_lo, y_hi = min(ys) - M36_CELL, max(ys) + M36_CELL
    i_lo = int(np.floor((x_lo - X_ORIGIN) / M36_CELL))
    i_hi = int(np.ceil( (x_hi - X_ORIGIN) / M36_CELL))
    j_lo = int(np.floor((Y_ORIGIN - y_hi) / M36_CELL))
    j_hi = int(np.ceil( (Y_ORIGIN - y_lo) / M36_CELL))
    x_bounds = X_ORIGIN + np.arange(i_lo, i_hi + 1) * M36_CELL
    y_bounds = Y_ORIGIN - np.arange(j_lo, j_hi + 1) * M36_CELL
    xs_samp = np.linspace(x_lo, x_hi, 50)
    ys_samp = np.linspace(y_lo, y_hi, 50)
    kw = dict(color='#aaa', lw=0.5, transform=ccrs.PlateCarree(), zorder=2)
    for xb in x_bounds:
        lo, la = EASE2(np.full(50, xb), ys_samp, inverse=True)
        ax.plot(lo, la, **kw)
    for yb in y_bounds:
        lo, la = EASE2(xs_samp, np.full(50, yb), inverse=True)
        ax.plot(lo, la, **kw)


def setup_map(ax, row, col, nrows):
    pad = 0.05
    ax.set_extent([LON0 - pad, LON1 + pad, LAT0 - pad, LAT1 + pad], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#f0f0f0')
    ax.add_feature(cfeature.STATES.with_scale('10m'), linewidth=0.5,
                   edgecolor='#bbb', facecolor='none')
    draw_m36_grid(ax)
    gl = ax.gridlines(draw_labels=True, linewidth=0, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = (row == nrows - 1)
    gl.left_labels = (col == 0)


def select_window(obs, window):
    """Return obs dict subset for one GEOSldas 3-hour assimilation window."""
    keep = np.asarray(obs['window']) == window
    return {k: np.asarray(v)[keep] for k, v in obs.items()}


def select_super_window(super_obs, window):
    """Return super-ob dict subset for one GEOSldas 3-hour assimilation window."""
    keep = np.asarray(super_obs['window']) == window
    return {k: np.asarray(v)[keep] for k, v in super_obs.items()}


In [ ]:
# Combined spatial maps for useful date/cycle windows across the 10-day sample.
VMIN, VMAX = 0, 100
cmap_ssm = plt.cm.YlOrBr
norm_ssm = mcolors.Normalize(VMIN, VMAX)

nrows = len(PLATFORMS)
col_titles = ['Legacy obs', 'H121 obs', 'Legacy tile-window', 'H121 tile-window']

selection_rows = []
for date, payload in daily.items():
    raw_day = payload['raw']
    so_day = payload['so']
    for plot_window in PLOT_WINDOWS:
        platform_rows = []
        for plat in PLATFORMS:
            legacy_obs = int(np.sum(np.asarray(raw_day[plat]['bufr']['window']) == plot_window))
            h121_obs = int(np.sum(np.asarray(raw_day[plat]['h121']['window']) == plot_window))
            legacy_tiles = int(np.sum(np.asarray(so_day[plat]['bufr_win']['window']) == plot_window))
            h121_tiles = int(np.sum(np.asarray(so_day[plat]['h121_win']['window']) == plot_window))
            platform_rows.append({
                'platform': plat,
                'legacy_obs': legacy_obs,
                'h121_obs': h121_obs,
                'legacy_tiles': legacy_tiles,
                'h121_tiles': h121_tiles,
                'max_product_obs': max(legacy_obs, h121_obs),
            })
        best = max(platform_rows, key=lambda r: r['max_product_obs'])
        if best['max_product_obs'] >= MIN_SPATIAL_MAP_OBS:
            selection_rows.append({
                'date': date,
                'window': int(plot_window),
                'cycle_utc': f'{3*int(plot_window):02d}00z',
                'best_platform': best['platform'],
                'max_product_obs': best['max_product_obs'],
                'legacy_obs_total': sum(r['legacy_obs'] for r in platform_rows),
                'h121_obs_total': sum(r['h121_obs'] for r in platform_rows),
                'legacy_tiles_total': sum(r['legacy_tiles'] for r in platform_rows),
                'h121_tiles_total': sum(r['h121_tiles'] for r in platform_rows),
            })

spatial_map_windows = pd.DataFrame(selection_rows).sort_values(['date', 'window']).reset_index(drop=True)
if MAX_SPATIAL_MAP_FIGURES is not None:
    spatial_map_windows = spatial_map_windows.head(MAX_SPATIAL_MAP_FIGURES)

if len(spatial_map_windows) == 0:
    print(f'No date/cycle windows met MIN_SPATIAL_MAP_OBS={MIN_SPATIAL_MAP_OBS}.')
else:
    display(spatial_map_windows.assign(date=spatial_map_windows['date'].dt.date))
    print(f'Plotting {len(spatial_map_windows)} date/cycle windows with at least {MIN_SPATIAL_MAP_OBS} obs for one platform/product.')

for _, row in spatial_map_windows.iterrows():
    date = row['date']
    plot_window = int(row['window'])
    raw_day = daily[date]['raw']
    so_day = daily[date]['so']

    fig, axes = plt.subplots(nrows, 4, figsize=(18, 4.2 * nrows),
                             subplot_kw={'projection': ccrs.PlateCarree()})
    axes = np.atleast_2d(axes)
    window_label = f'GEOS cycle {3*plot_window:02d}00z'

    for i, (plat, cfg) in enumerate(PLATFORMS.items()):
        b = select_window(raw_day[plat]['bufr'], plot_window)
        h = select_window(raw_day[plat]['h121'], plot_window)

        for j in range(4):
            setup_map(axes[i, j], i, j, nrows)
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=10, fontweight='bold')

        axes[i, 0].text(-0.16, 0.5, plat, transform=axes[i, 0].transAxes,
                        fontsize=10, fontweight='bold', va='center', rotation=90)

        # Raw legacy observations for the selected window.
        if len(b['ssm']):
            axes[i, 0].scatter(b['lon'], b['lat'], c=b['ssm'], cmap=cmap_ssm,
                               norm=norm_ssm, s=40, marker='s',
                               transform=ccrs.PlateCarree(), zorder=4)
        axes[i, 0].text(0.02, 0.98, f'{len(b["ssm"])} obs', transform=axes[i, 0].transAxes,
                        ha='left', va='top', fontsize=8,
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))

        # Raw H121 observations for the selected window.
        if len(h['ssm']):
            axes[i, 1].scatter(h['lon'], h['lat'], c=h['ssm'], cmap=cmap_ssm,
                               norm=norm_ssm, s=10,
                               transform=ccrs.PlateCarree(), zorder=4)
        axes[i, 1].text(0.02, 0.98, f'{len(h["ssm"])} obs', transform=axes[i, 1].transAxes,
                        ha='left', va='top', fontsize=8,
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))

        # Super-ob tiles for the selected window.
        for j, key in zip([2, 3], ['bufr_win', 'h121_win']):
            s = select_super_window(so_day[plat][key], plot_window)
            for k in range(len(s['ij'])):
                ci, cj = s['ij'][k]
                clons, clats = ij_to_corners(ci, cj)
                color = cmap_ssm(norm_ssm(s['ssm'][k]))
                poly = mpatches.Polygon(
                    np.column_stack([clons, clats]),
                    facecolor=color, edgecolor='#555', linewidth=0.6, alpha=0.85,
                    transform=ccrs.PlateCarree(), zorder=3)
                axes[i, j].add_patch(poly)
                axes[i, j].text(np.mean(clons), np.mean(clats), str(s['count'][k]),
                                ha='center', va='center', fontsize=6, fontweight='bold',
                                transform=ccrs.PlateCarree(), zorder=5)

            raw_obs = b if key == 'bufr_win' else h
            axes[i, j].text(0.02, 0.98,
                            f'{len(raw_obs["ssm"])} obs -> {len(s["ssm"])} GEOS tile-windows',
                            transform=axes[i, j].transAxes,
                            ha='left', va='top', fontsize=8,
                            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2))

    fig.subplots_adjust(right=0.91, hspace=0.08, wspace=0.08)
    cax = fig.add_axes([0.93, 0.16, 0.012, 0.68])
    fig.colorbar(plt.cm.ScalarMappable(norm=norm_ssm, cmap=cmap_ssm), cax=cax, label='SSM (% sat)')
    fig.suptitle(
        f'Raw observations and GEOS tile super-obs by analysis cycle  |  '
        f'{date.date()} {window_label}  |  {LAT0}-{LAT1} deg N, {LON0}-{LON1} deg E',
        y=1.01,
    )
    plt.show()


## 8. Global scatter: Legacy vs H121 super-obs on matched tiles

Compare Legacy and H121 in the same space GEOSldas uses: global M36 tile/cycle super-obs over the full 10-day sample. Matching is by analysis date, platform, `tilenum`, and GEOS cycle.


In [ ]:
def global_superob_file(date):
    return GLOBAL_SUPEROB_CACHE_DIR / f'ascat_global_superobs_{date:%Y%m%d}_{GLOBAL_SUPEROB_VERSION}.pkl'


def normalize_global_superobs(df):
    """Map source-file dates to GEOS analysis dates and collapse split cycles."""
    out = df.copy()
    source_date = pd.to_datetime(out['date'])
    cycle = out['cycle'].astype('int16')
    out['date'] = (source_date + pd.to_timedelta((cycle // 8).astype(int), unit='D')).dt.strftime('%Y-%m-%d')
    out['cycle'] = (cycle % 8).astype('int16')
    out['window'] = out['cycle'].astype('int8')

    n_obs = out['n_obs'].astype(float)
    out['_ssm_sum'] = out['ssm_pct'] * n_obs
    out['_lat_sum'] = out['lat'] * n_obs
    out['_lon_sum'] = out['lon'] * n_obs
    out['_ssm2_sum'] = (out['ssm_std_pct'] ** 2 + out['ssm_pct'] ** 2) * n_obs

    grouped = (
        out.groupby(['date', 'product', 'platform', 'tilenum', 'cycle'], as_index=False)
        .agg(
            window=('window', 'first'),
            i_indg=('i_indg', 'first'),
            j_indg=('j_indg', 'first'),
            n_obs=('n_obs', 'sum'),
            _ssm_sum=('_ssm_sum', 'sum'),
            _lat_sum=('_lat_sum', 'sum'),
            _lon_sum=('_lon_sum', 'sum'),
            _ssm2_sum=('_ssm2_sum', 'sum'),
            ssm_min_pct=('ssm_min_pct', 'min'),
            ssm_max_pct=('ssm_max_pct', 'max'),
        )
    )
    n = grouped['n_obs'].astype(float)
    grouped['ssm_pct'] = grouped['_ssm_sum'] / n
    grouped['lat'] = grouped['_lat_sum'] / n
    grouped['lon'] = grouped['_lon_sum'] / n
    var = grouped['_ssm2_sum'] / n - grouped['ssm_pct'] ** 2
    grouped['ssm_std_pct'] = np.sqrt(np.maximum(var, 0.0))
    return grouped.drop(columns=['_ssm_sum', '_lat_sum', '_lon_sum', '_ssm2_sum'])


frames = []
missing = []
for date in DATES:
    f = global_superob_file(date)
    if f.exists():
        frames.append(pd.read_pickle(f))
    else:
        missing.append(f)

if missing:
    raise FileNotFoundError('Missing global super-ob cache files: ' + ', '.join(str(f) for f in missing))

global_superobs = normalize_global_superobs(pd.concat(frames, ignore_index=True))
start_str = DATES[0].strftime('%Y-%m-%d')
end_str = DATES[-1].strftime('%Y-%m-%d')
global_superobs = global_superobs[(global_superobs['date'] >= start_str) & (global_superobs['date'] <= end_str)].copy()

legacy_global = global_superobs[global_superobs['product'] == 'legacy'].rename(columns={
    'ssm_pct': 'legacy_ssm_pct',
    'n_obs': 'legacy_n_obs',
    'ssm_std_pct': 'legacy_std_pct',
    'ssm_min_pct': 'legacy_min_pct',
    'ssm_max_pct': 'legacy_max_pct',
})
h121_global = global_superobs[global_superobs['product'] == 'h121'].rename(columns={
    'ssm_pct': 'h121_ssm_pct',
    'n_obs': 'h121_n_obs',
    'ssm_std_pct': 'h121_std_pct',
    'ssm_min_pct': 'h121_min_pct',
    'ssm_max_pct': 'h121_max_pct',
})

match_keys = ['date', 'platform', 'tilenum', 'cycle']
global_matched = legacy_global.merge(
    h121_global,
    on=match_keys,
    suffixes=('_legacy', '_h121'),
    how='inner',
)
global_matched['diff_pct'] = global_matched['h121_ssm_pct'] - global_matched['legacy_ssm_pct']

summary_rows = []
for plat in PLATFORMS:
    g = global_matched[global_matched['platform'] == plat]
    d = g['diff_pct'].to_numpy(float)
    summary_rows.append({
        'platform': plat,
        'matched_tile_cycles': len(g),
        'bias_pct': np.mean(d) if len(d) else np.nan,
        'rmse_pct': np.sqrt(np.mean(d ** 2)) if len(d) else np.nan,
        'mae_pct': np.mean(np.abs(d)) if len(d) else np.nan,
        'median_abs_pct': np.median(np.abs(d)) if len(d) else np.nan,
        'r': np.corrcoef(g['legacy_ssm_pct'], g['h121_ssm_pct'])[0, 1] if len(g) > 1 else np.nan,
        'legacy_n_obs_median': g['legacy_n_obs'].median() if len(g) else np.nan,
        'h121_n_obs_median': g['h121_n_obs'].median() if len(g) else np.nan,
    })

all_d = global_matched['diff_pct'].to_numpy(float)
summary_rows.append({
    'platform': 'All',
    'matched_tile_cycles': len(global_matched),
    'bias_pct': np.mean(all_d) if len(all_d) else np.nan,
    'rmse_pct': np.sqrt(np.mean(all_d ** 2)) if len(all_d) else np.nan,
    'mae_pct': np.mean(np.abs(all_d)) if len(all_d) else np.nan,
    'median_abs_pct': np.median(np.abs(all_d)) if len(all_d) else np.nan,
    'r': np.corrcoef(global_matched['legacy_ssm_pct'], global_matched['h121_ssm_pct'])[0, 1] if len(global_matched) > 1 else np.nan,
    'legacy_n_obs_median': global_matched['legacy_n_obs'].median() if len(global_matched) else np.nan,
    'h121_n_obs_median': global_matched['h121_n_obs'].median() if len(global_matched) else np.nan,
})

global_match_summary = pd.DataFrame(summary_rows)
num_cols = ['bias_pct', 'rmse_pct', 'mae_pct', 'median_abs_pct', 'r', 'legacy_n_obs_median', 'h121_n_obs_median']
global_match_summary[num_cols] = global_match_summary[num_cols].round(3)
display(global_match_summary)


In [ ]:
SCATTER_SAMPLE_PER_PLATFORM = 120000

plot_parts = []
for plat, group in global_matched.groupby('platform'):
    plot_parts.append(group.sample(min(len(group), SCATTER_SAMPLE_PER_PLATFORM), random_state=42))
global_matched_plot = pd.concat(plot_parts, ignore_index=True) if plot_parts else pd.DataFrame()

fig, axes = plt.subplots(1, len(PLATFORMS) + 1, figsize=(16, 4.5), sharey=True, sharex=True)
ref = np.array([0, 100])

for ax, (plat, cfg) in zip(axes[:-1], PLATFORMS.items()):
    p = global_matched_plot[global_matched_plot['platform'] == plat]
    s = global_match_summary[global_match_summary['platform'] == plat].iloc[0]
    color = cfg['color']
    ax.plot(ref, ref, 'k--', lw=0.8, zorder=1)
    if len(p):
        ax.scatter(p['legacy_ssm_pct'], p['h121_ssm_pct'], color=color, s=2, alpha=0.15, zorder=3)
    txt = '\n'.join([
        f"n={int(s['matched_tile_cycles']):,}",
        f"bias={s['bias_pct']:+.2f}%",
        f"RMSE={s['rmse_pct']:.2f}%",
        f"med abs={s['median_abs_pct']:.2f}%",
        f"r={s['r']:.2f}",
    ])
    ax.text(0.04, 0.97, txt, transform=ax.transAxes, fontsize=8, va='top', color=color,
            bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2))
    ax.set_title(plat, fontsize=10)
    ax.set_xlabel('Legacy BUFR super-ob (% sat)', fontsize=9)
    ax.set_aspect('equal')

axes[0].set_ylabel('H121 super-ob (% sat)', fontsize=9)

ax = axes[-1]
ax.plot(ref, ref, 'k--', lw=0.8, zorder=1)
for plat, cfg in PLATFORMS.items():
    p = global_matched_plot[global_matched_plot['platform'] == plat]
    if len(p):
        ax.scatter(p['legacy_ssm_pct'], p['h121_ssm_pct'], color=cfg['color'], s=2, alpha=0.13,
                   label=plat, zorder=3)
s = global_match_summary[global_match_summary['platform'] == 'All'].iloc[0]
txt = '\n'.join([
    f"n={int(s['matched_tile_cycles']):,}",
    f"bias={s['bias_pct']:+.2f}%",
    f"RMSE={s['rmse_pct']:.2f}%",
    f"med abs={s['median_abs_pct']:.2f}%",
    f"r={s['r']:.2f}",
])
ax.text(0.04, 0.97, txt, transform=ax.transAxes, fontsize=8, va='top', color='k',
        bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2))
ax.set_title('All platforms', fontsize=10)
ax.set_xlabel('Legacy BUFR super-ob (% sat)', fontsize=9)
ax.legend(fontsize=8, loc='lower right')
ax.set_aspect('equal')

for ax in axes:
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)

fig.suptitle(
    f'Global Legacy vs H121 GEOS tile/cycle super-obs  |  {DATES[0].date()} to {DATES[-1].date()}  |  sampled points, exact stats',
    y=1.03,
)
fig.tight_layout()
plt.show()
